# Theta phase precession from button presses

This notebook estimates theta phase precession from the UPPT002 button-press times and the MEG epochs. It filters each MEG sensor to 2–5 Hz, extracts phase with MNE's Hilbert transform, then uses circ_regress_db to fit a circular-linear slope and signed correlation for each trial and sensor.

Trial results are averaged within each participant before the participant means are combined. Subject 11 is excluded. Only trials with all five valid press times are included. The group permutation tests use participant means and apply false-discovery-rate correction across sensors; they are not the per-trial phase-shuffle tests from circ_regress_db.

import packages for loading and processing MEG epochs, circular regression, tables and figures

In [ ]:
import os
import mne
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from circ_regress_db import circ_regress_db

define the data locations and analysis parameters

MEG_DATA_ROOT can point to a local MEG export; otherwise the notebook uses the repository's network-share path. The press-time file is expected in the repository folder.

theta_freq_band sets the filter limits. Subject 11 is excluded before analysis. CHANNEL_BATCH_SIZE limits memory use while channels are filtered and Hilbert-transformed. N_GROUP_PERMUTATIONS sets the number of permutations for the participant-level tests.

In [ ]:
DATA_ROOT = os.environ.get(
    "MEG_DATA_ROOT",
    r"//rdp.arc.ucl.ac.uk/ritd-ag-project-rd01pz-dbush99/Katja Button Press Data"
)
BUTTON_PRESS_TIMES_FILE = os.path.join(
    os.getcwd(), "UPPT002_button_press_times.npz"
)

theta_freq_band = [2.0, 5.0]         # theta band edges in Hz
excluded_subjects = ["11"]           # participants excluded from analysis
CHANNEL_BATCH_SIZE = 8               # limit memory while filtering/Hilbert-transforming
N_GROUP_PERMUTATIONS = 5000           # group-level permutation tests

load and check the UPPT002 button-press times

The NPZ file stores uppt002_times in participant × trial × press-slot order. subject_ID records the participant row order, n_trials_by_subject gives each participant's recorded trial count, and press_slot labels presses 1 through 5. Times are in seconds relative to cue onset.

The ID helper normalizes participant labels to the repository's two-digit format. The checks confirm the time reference, array dimensions and press-slot order before the participant list is made. Subject 11 is omitted from sub_ID here, so it is not included in the per-participant or group calculations.

In [ ]:
def normalise_subject_id(id):
    return str(int(str(id).strip().lower().lstrip("s"))).zfill(2)


with np.load(BUTTON_PRESS_TIMES_FILE, allow_pickle=False) as press_file:
    UPPT002_times = np.asarray(press_file["uppt002_times"], dtype=float)
    subject_ID = np.asarray(
        [normalise_subject_id(id) for id in press_file["subject_ID"]]
    )
    n_trials_by_subject = np.asarray(
        press_file["n_trials_by_subject"], dtype=int
    )
    press_slot = np.asarray(press_file["press_slot"], dtype=int)
    CUE_ONSET_AFTER_EPOCH_START_S = float(
        np.asarray(press_file["cue_onset_after_epoch_start_s"]).item()
    )
    time_reference = str(press_file["time_reference"].item())

if "relative to cue onset" not in time_reference.lower():
    raise ValueError("Expected button-press times in seconds relative to cue onset")
if UPPT002_times.ndim != 3 or UPPT002_times.shape[0] != len(subject_ID):
    raise ValueError("Unexpected participant × trial × press-slot array shape")
if not np.array_equal(press_slot, np.arange(1, len(press_slot) + 1)):
    raise ValueError("Expected press slots numbered in sequence from 1")

n_press_slots = len(press_slot)
sub_ID = [id for id in subject_ID if id not in excluded_subjects]
subject_index = {id: index for index, id in enumerate(subject_ID)}
if not sub_ID:
    raise ValueError("No participants remain after applying excluded_subjects.")

print("Array shape:", UPPT002_times.shape)
print("Participants:", sub_ID)
print("Excluded participants:", excluded_subjects)

load each participant's MEG epochs

filename_loader follows the repository layout: epochs are under sXX and the filename uses the participant number without its leading zero. It also checks for a local export with the FIF file at the data-root level.

The epochs are opened without loading all channels into memory. MEG_sens contains channels whose names begin with M. mean_finite averages finite values along the requested axis and returns NaN when no values are available.

In [ ]:
def filename_loader(id):
    subject_number = str(int(id))
    subject_folder = "s" + id
    file_name = "Subject" + subject_number + "_spm_mne-epo.fif"
    epochs_file = os.path.join(DATA_ROOT, subject_folder, file_name)

    if not os.path.isfile(epochs_file):
        epochs_file = os.path.join(DATA_ROOT, file_name)
    if not os.path.isfile(epochs_file):
        raise FileNotFoundError("Could not find " + file_name + " under " + DATA_ROOT)

    epochs = mne.read_epochs(epochs_file, preload=False, verbose="ERROR")
    MEG_sens = [channel for channel in epochs.ch_names if channel.startswith("M")]
    if not MEG_sens:
        raise ValueError("No MEG sensors found in " + epochs_file)

    return epochs, MEG_sens


def mean_finite(values, axis=0):
    values = np.asarray(values)
    counts = np.isfinite(values).sum(axis=axis)
    return np.divide(
        np.nansum(values, axis=axis),
        counts,
        out=np.full(counts.shape, np.nan, dtype=np.result_type(values.dtype, float)),
        where=counts > 0,
    )

calculate theta phase precession for one participant

The function checks that the NPZ timing rows match the saved MEG epochs, maps cue-relative press times onto the epoch time axis, and marks trials with all five valid press times.

For each small sensor batch, get_data reads only the requested channels from the non-preloaded epochs. The batch is wrapped in an EpochsArray with matching sensor information; MNE then filters it to theta and applies the Hilbert transform. This keeps the original epochs on disk and loads only the current channel batch into memory.

The phase at each press is passed to circ_regress_db, which fits the slope and phase offset for the press-slot sequence and returns the signed circular correlation. The function averages slope and correlation across valid trials, averages phase offsets circularly, and returns sensor geometry and trial counts for review.

In [ ]:
        channel_batch = MEG_sens[start:start + CHANNEL_BATCH_SIZE]
        channel_indices = mne.pick_channels(
            epochs.ch_names, include=channel_batch
        )
        channel_data = epochs.get_data(picks=channel_indices)
        batch_info = mne.pick_info(epochs.info, channel_indices, copy=True)

        # Load only this channel batch into an EpochsArray for MNE processing.
        epochs_batch = mne.EpochsArray(
            channel_data,
            batch_info,
            tmin=epochs.tmin,
            baseline=None,
            verbose="ERROR",
        )
        epochs_batch.filter(
            l_freq=theta_freq_band[0],
            h_freq=theta_freq_band[1],
            verbose="ERROR",
        ).apply_hilbert(envelope=False, verbose="ERROR")
        analytic = epochs_batch.get_data()

process the included participants

sub_ID already excludes subject 11. Each participant is analyzed separately, and the sensor names and order are checked before values are combined. This prevents channel values from being averaged under mismatched sensor labels.

The group values are means of the participant means, so participants with more valid trials do not receive extra weight. Phase offsets are averaged as circular values.

In [ ]:
participant_slope = []
participant_corr = []
participant_phase = []
participant_rows = []
group_sensors = None
group_info = None

for id in sub_ID:
    slope, corr, phase, MEG_sens, sensor_info, summary = (
        participant_phase_precession(id)
    )
    if group_sensors is None:
        group_sensors, group_info = MEG_sens, sensor_info
    elif MEG_sens != group_sensors:
        raise ValueError("MEG sensor order differs for participant " + id)

    participant_slope.append(slope)
    participant_corr.append(corr)
    participant_phase.append(phase)
    participant_rows.append(summary)
    print(
        "Participant", id,
        "- complete trials:", summary["n_complete_press_trials"],
    )

participant_slope = np.asarray(participant_slope)
participant_corr = np.asarray(participant_corr)
participant_phase = np.asarray(participant_phase)
participant_summary_table = pd.DataFrame(participant_rows)

group_slope = mean_finite(participant_slope, axis=0)
group_corr = mean_finite(participant_corr, axis=0)
group_phase = np.angle(
    mean_finite(np.exp(1j * participant_phase), axis=0)
)

test participant means and review the results

For each sensor, group_permutation_test runs a two-sided one-sample permutation test across participant means against zero. The FDR correction is applied across sensors separately for correlation and slope.

The tables show participant-level sensor values, participant trial counts, group means and corrected p-values. These p-values describe group-level effects across participants; they are not the per-trial phase-shuffle p-values returned by circ_regress_db when k is greater than zero.

In [ ]:
def group_permutation_test(values):
    # Test participant means against zero; correct across sensors with FDR.
    t_value = np.full(values.shape[1], np.nan)
    p_value = np.full(values.shape[1], np.nan)
    rng = np.random.RandomState(0)

    for sensor_index in range(values.shape[1]):
        sensor_values = values[:, sensor_index]
        sensor_values = sensor_values[np.isfinite(sensor_values)]
        if len(sensor_values) < 3:
            continue

        t_value[sensor_index], p_value[sensor_index], _ = (
            mne.stats.permutation_t_test(
                sensor_values[:, None],
                n_permutations=N_GROUP_PERMUTATIONS,
                tail=0,
                seed=rng,
                verbose="ERROR",
            )
        )

    p_fdr = np.full_like(p_value, np.nan)
    valid = np.isfinite(p_value)
    if valid.any():
        _, p_fdr[valid] = mne.stats.fdr_correction(p_value[valid], alpha=0.05)

    return t_value, p_value, p_fdr


corr_t, corr_p, corr_p_fdr = group_permutation_test(participant_corr)
slope_t, slope_p, slope_p_fdr = group_permutation_test(participant_slope)

participant_corr_table = pd.DataFrame(
    participant_corr, index=sub_ID, columns=group_sensors
)
participant_slope_table = pd.DataFrame(
    participant_slope, index=sub_ID, columns=group_sensors
)
participant_phase_table = pd.DataFrame(
    participant_phase, index=sub_ID, columns=group_sensors
)

group_results = pd.DataFrame({
    "sensor": group_sensors,
    "mean_corr": group_corr,
    "corr_t": corr_t,
    "corr_p": corr_p,
    "corr_p_fdr": corr_p_fdr,
    "mean_slope_rad_per_press": group_slope,
    "slope_t": slope_t,
    "slope_p": slope_p,
    "slope_p_fdr": slope_p_fdr,
    "mean_phase_offset_rad": group_phase,
})

display(participant_summary_table.round(4))
display(participant_corr_table.round(3))
display(participant_slope_table.round(3))
display(group_results.round(4))

plot the group mean for each MEG sensor type

The correlation and slope maps show the equal-weight mean across participant means. Magnetometers and gradiometers are plotted separately using the channel geometry loaded from the FIF epochs.

The displayed color limits span the full coefficient or slope range. Use group_results to review the corresponding uncorrected and FDR-corrected sensor-level p-values.

In [ ]:
sensor_types = [
    mne.channel_type(group_info, index)
    for index in range(len(group_sensors))
]

for values, label, limits in [
    (group_corr, "Mean signed circular correlation", (-1.0, 1.0)),
    (group_slope, "Mean slope (radians per press)", (-2 * np.pi, 2 * np.pi)),
]:
    for sensor_type in sorted(set(sensor_types)):
        picks = np.array([
            index for index, kind in enumerate(sensor_types)
            if kind == sensor_type
        ])
        sensor_info = mne.pick_info(group_info, picks, copy=True)
        fig, ax = plt.subplots(figsize=(7, 6))
        image, _ = mne.viz.plot_topomap(
            values[picks],
            sensor_info,
            ch_type=sensor_type,
            axes=ax,
            cmap="RdBu_r",
            contours=0,
            show=False,
        )
        image.set_clim(*limits)
        ax.set_title(label + " (" + sensor_type + ")")
        fig.colorbar(image, ax=ax)
        plt.tight_layout()
        plt.show()